# Notebook 03 — Custom-50 Binary Evaluation
**D7047E Advanced Deep Learning | Group 14**

Evaluates the best binary model (IAM test F1) on the **custom-50** real-photo dataset (CLEAN vs CROSSED-OUT).

Generalisation gap = IAM test F1 − Custom-50 F1.

**Prerequisite:** Run `01_simplecnn_binary.ipynb` (or `02_binary.ipynb`) to produce `best_binary_*.pth` checkpoints.

WandB: `adl-crossout-v5 / custom_binary / {model_name}`

## 1. Colab / Drive Setup

In [ ]:
import os
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ
DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MOUNTED:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
    except Exception as e:
        print(f'Drive mount skipped ({e}).')
print(f'IN_COLAB={IN_COLAB}  DRIVE_MOUNTED={DRIVE_MOUNTED}')

## 2. Configuration

In [ ]:
import sys, os
sys.path.insert(0, '..')

IN_COLAB      = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ
DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')

DATA_DIR   = '../dataset/iam_crossouts'
CUSTOM_DIR = '../dataset/custom_50'

if DRIVE_MOUNTED:
    CHECKPOINT_DIR = '/content/drive/MyDrive/adl_checkpoints'
elif IN_COLAB:
    CHECKPOINT_DIR = '/content/adl_checkpoints'
else:
    CHECKPOINT_DIR = '../checkpoints'

IMG_SIZE    = 224
BATCH_SIZE  = 64
NUM_WORKERS = 4
WANDB_GROUP = 'custom_binary'

print(f'Config loaded. CHECKPOINT_DIR={CHECKPOINT_DIR}')

## 3. Setup

In [ ]:
!pip install -q gdown torch torchvision pillow matplotlib scikit-learn wandb python-dotenv

In [ ]:
import os
os.environ['WANDB_API_KEY'] = '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'

In [ ]:
import os, gc
import torch
from torch.utils.data import DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score)
import wandb
from dotenv import load_dotenv

from common import (get_transforms, WANDB_PROJECT, BinaryDataset, rebuild_binary_model)

BINARY_LABELS = ['CLEAN', 'CROSSED']

load_dotenv()
wandb.login(key=os.environ.get('WANDB_API_KEY'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 4. Load All Binary Checkpoints & Pick Best (IAM Test)

In [ ]:
_, val_t = get_transforms(IMG_SIZE)
ldr_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=True, prefetch_factor=2 if NUM_WORKERS > 0 else None)

bin_test_ds     = BinaryDataset(os.path.join(DATA_DIR, 'test', 'images'), val_t)
bin_test_loader = DataLoader(bin_test_ds, shuffle=False, **ldr_kw)

bin_files = [f for f in os.listdir(CHECKPOINT_DIR)
             if f.startswith('best_binary_') and f.endswith('.pth')]
print(f'Found {len(bin_files)} binary checkpoints: {bin_files}')

if not bin_files:
    raise FileNotFoundError(
        f'No binary checkpoints found in {CHECKPOINT_DIR}.\n'
        'Run 01_simplecnn_binary.ipynb or 02_binary.ipynb first to train and save binary models.'
    )

bin_table = {}
for fname in sorted(bin_files):
    path = os.path.join(CHECKPOINT_DIR, fname)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model_name = ckpt['model_name']
    model = rebuild_binary_model(model_name)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()

    preds, labels, probs = [], [], []
    with torch.no_grad():
        for imgs, lbs in bin_test_loader:
            out = model(imgs.to(device))
            probs.extend(torch.softmax(out, dim=1)[:, 1].cpu().tolist())
            preds.extend(out.argmax(1).cpu().tolist())
            labels.extend(lbs.tolist())

    f1 = f1_score(labels, preds, zero_division=0)
    bin_table[model_name] = {'f1': f1, 'preds': preds, 'labels': labels, 'probs': probs}
    print(f'  {model_name:<18} F1={f1:.4f}  (checkpoint: {fname})')
    del model; torch.cuda.empty_cache(); gc.collect()

best_bin_name = max(bin_table, key=lambda k: bin_table[k]['f1'])
print(f'\nBest binary model (IAM test): {best_bin_name}  F1={bin_table[best_bin_name]["f1"]:.4f}')

## 5. Custom-50 Binary Evaluation

In [ ]:
bin_custom_ds     = BinaryDataset(CUSTOM_DIR, val_t)
bin_custom_loader = DataLoader(bin_custom_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'Custom-50 binary: {len(bin_custom_ds)} images\n')

bin_c50_results = {}
for model_name in sorted(bin_table.keys()):
    safe = model_name.replace('/', '_').replace('-', '_')
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'best_binary_{safe}.pth')
    model = rebuild_binary_model(model_name)
    model.load_state_dict(
        torch.load(ckpt_path, map_location=device, weights_only=False)['model_state_dict']
    )
    model = model.to(device).eval()

    preds, labels, probs = [], [], []
    with torch.no_grad():
        for imgs, lbs in bin_custom_loader:
            out = model(imgs.to(device))
            probs.extend(torch.softmax(out, dim=1)[:, 1].cpu().tolist())
            preds.extend(out.argmax(1).cpu().tolist())
            labels.extend(lbs.tolist())

    c50_f1  = f1_score(labels, preds, zero_division=0)
    c50_acc = accuracy_score(labels, preds)
    bin_c50_results[model_name] = {
        'f1': c50_f1, 'acc': c50_acc,
        'preds': preds, 'labels': labels, 'probs': probs
    }
    print(f'  {model_name:<18} Acc={c50_acc:.4f}  F1={c50_f1:.4f}')
    del model; torch.cuda.empty_cache(); gc.collect()

best_bin_c50_name = max(bin_c50_results, key=lambda k: bin_c50_results[k]['f1'])
print(f'\nBest on Custom-50: {best_bin_c50_name}')
print()
print(classification_report(
    bin_c50_results[best_bin_c50_name]['labels'],
    bin_c50_results[best_bin_c50_name]['preds'],
    target_names=BINARY_LABELS, zero_division=0
))

## 6. Generalisation Gap & WandB Logging

In [ ]:
print('=== Generalisation Gap — Binary ===')
print(f'{"Model":<18} {"IAM Test F1":>12} {"Custom-50 F1":>13} {"Gap":>8}')
print('-' * 55)
for name in sorted(bin_table.keys()):
    iam = bin_table[name]['f1']
    c50 = bin_c50_results[name]['f1']
    marker = ' <-- best' if name == best_bin_c50_name else ''
    print(f'{name:<18} {iam:>12.4f} {c50:>13.4f} {iam - c50:>+8.4f}{marker}')

print('\nLogging to WandB...')
for name in bin_table:
    r = bin_c50_results[name]
    run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f'binary_{name}',
                     config=dict(model=name, task='custom50_binary'), reinit=True)
    try:
        auc = roc_auc_score(r['labels'], r['probs'])
    except Exception:
        auc = float('nan')
    cm = confusion_matrix(r['labels'], r['preds'])
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm, display_labels=BINARY_LABELS).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Custom-50 Binary CM — {name}')
    plt.tight_layout()
    wandb.log({
        'iam_test_f1':        bin_table[name]['f1'],
        'custom50_acc':       r['acc'],
        'custom50_f1':        r['f1'],
        'custom50_precision': precision_score(r['labels'], r['preds'], zero_division=0),
        'custom50_recall':    recall_score(r['labels'], r['preds'], zero_division=0),
        'custom50_auc':       auc,
        'generalisation_gap': bin_table[name]['f1'] - r['f1'],
        'confusion_matrix':   wandb.Image(fig),
    })
    plt.close('all')
    run.finish()
print('All results logged to WandB.')

## 7. Comparison Bar Chart — IAM Test vs Custom-50

In [ ]:
bin_names = sorted(bin_table.keys())
x = np.arange(len(bin_names))
w = 0.35

bin_iam = [bin_table[n]['f1']       for n in bin_names]
bin_c50 = [bin_c50_results[n]['f1'] for n in bin_names]

fig, ax = plt.subplots(figsize=(max(6, len(bin_names) * 2), 5))
bars1 = ax.bar(x - w/2, bin_iam, w, label='IAM Test',  color='#5c85d6')
bars2 = ax.bar(x + w/2, bin_c50, w, label='Custom-50', color='#e07b39')
ax.set_xticks(x)
ax.set_xticklabels(bin_names, rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('F1')
ax.set_title('Binary — IAM Test vs Custom-50 Generalisation')
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('custom50_binary_comparison.png', dpi=150)
plt.show()
print('Saved: custom50_binary_comparison.png')

## 8. Confusion Matrix — Best Binary Model

In [ ]:
r = bin_c50_results[best_bin_c50_name]
cm = confusion_matrix(r['labels'], r['preds'])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=BINARY_LABELS).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Custom-50 Binary CM — {best_bin_c50_name}')
plt.tight_layout()
plt.savefig('custom50_binary_cm.png', dpi=150)
plt.show()
print('Saved: custom50_binary_cm.png')

## 9. Sample Predictions Grid — Best Model (CLEAN vs CROSSED)

In [ ]:
from common import BINARY_CROSSED
from PIL import Image

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])
N_SAMPLES = 4

safe = best_bin_c50_name.replace('/', '_').replace('-', '_')
best_model = rebuild_binary_model(best_bin_c50_name)
best_model.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, f'best_binary_{safe}.pth'),
               map_location=device, weights_only=False)['model_state_dict']
)
best_model = best_model.to(device).eval()

# Sample N_SAMPLES images from CLEAN (label 0) and each crossed style (label 1)
sample_groups = [('CLEAN', 0)] + [(cat, 1) for cat in BINARY_CROSSED if
                                   os.path.exists(os.path.join(CUSTOM_DIR, cat))]
rows = len(sample_groups)
fig, axes = plt.subplots(rows, N_SAMPLES, figsize=(N_SAMPLES * 3, rows * 2))
if rows == 1: axes = [axes]

for row, (cat, true_lbl) in enumerate(sample_groups):
    folder = os.path.join(CUSTOM_DIR, cat)
    if not os.path.exists(folder):
        for col in range(N_SAMPLES): axes[row][col].axis('off')
        continue
    files = sorted([f for f in os.listdir(folder) if f.lower().endswith(('.png','.jpg','.jpeg'))])[:N_SAMPLES]
    for col, fname in enumerate(files):
        raw    = Image.open(os.path.join(folder, fname)).convert('RGB')
        tensor = val_t(raw).unsqueeze(0).to(device)
        with torch.no_grad():
            out      = best_model(tensor)
            prob_cross = torch.softmax(out, dim=1)[0, 1].item()
            pred_lbl   = out.argmax(1).item()
        pred_name = BINARY_LABELS[pred_lbl]
        disp = (val_t(raw) * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]).clamp(0, 1)
        axes[row][col].imshow(disp.permute(1, 2, 0).numpy())
        axes[row][col].axis('off')
        color = 'green' if pred_lbl == true_lbl else 'red'
        axes[row][col].set_title(f'{pred_name}\n{prob_cross:.0%}', fontsize=7, color=color)
    for col in range(len(files), N_SAMPLES):
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'GT: {cat}', fontsize=8, rotation=0, labelpad=65, va='center')

fig.suptitle(f'Custom-50 binary predictions — {best_bin_c50_name} (green=correct, red=wrong)', fontsize=11)
plt.tight_layout()
plt.savefig('custom50_binary_predictions.png', dpi=150)
plt.show()
print('Saved: custom50_binary_predictions.png')
del best_model; torch.cuda.empty_cache(); gc.collect()

## 10. Misclassified Examples — Best Model

In [ ]:
from common import BINARY_CROSSED
from PIL import Image

safe = best_bin_c50_name.replace('/', '_').replace('-', '_')
best_model = rebuild_binary_model(best_bin_c50_name)
best_model.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, f'best_binary_{safe}.pth'),
               map_location=device, weights_only=False)['model_state_dict']
)
best_model = best_model.to(device).eval()

all_classes = [('CLEAN', 0)] + [(cat, 1) for cat in BINARY_CROSSED]
wrong = []
total = 0
for cat, true_lbl in all_classes:
    folder = os.path.join(CUSTOM_DIR, cat)
    if not os.path.exists(folder):
        continue
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        total += 1
        img_path = os.path.join(folder, fname)
        raw    = Image.open(img_path).convert('RGB')
        tensor = val_t(raw).unsqueeze(0).to(device)
        with torch.no_grad():
            out      = best_model(tensor)
            pred_lbl = out.argmax(1).item()
            conf     = torch.softmax(out, dim=1)[0, pred_lbl].item()
        if pred_lbl != true_lbl:
            wrong.append((img_path, BINARY_LABELS[true_lbl], BINARY_LABELS[pred_lbl], conf, raw))

print(f'{best_bin_c50_name} — {len(wrong)} misclassified / {total} Custom-50 images\n')
for img_path, true_cls, pred_cls, conf, _ in wrong:
    print(f'  True: {true_cls:<8}  Pred: {pred_cls:<8}  Conf: {conf:.2f}  '
          f'File: {os.path.basename(img_path)}')

if wrong:
    cols = min(5, len(wrong))
    rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 3.4))
    axes = np.array(axes).flatten()

    for ax, (img_path, true_cls, pred_cls, conf, raw) in zip(axes, wrong):
        disp = (val_t(raw) * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]).clamp(0, 1)
        ax.imshow(disp.permute(1, 2, 0).numpy())
        ax.set_title(f'True:  {true_cls}\nPred:  {pred_cls}\nConf: {conf:.0%}',
                     fontsize=7, color='red')
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor('red'); spine.set_linewidth(2)

    for ax in axes[len(wrong):]:
        ax.set_visible(False)

    plt.suptitle(f'Misclassified — {best_bin_c50_name} on Custom-50 ({len(wrong)} errors)',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('custom50_binary_misclassified.png', dpi=150)
    plt.show()
    print('Saved: custom50_binary_misclassified.png')
else:
    print('No misclassified images — perfect score on Custom-50!')

del best_model; torch.cuda.empty_cache(); gc.collect()